# Rally 1

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murua-Alvaro/clases-y-otras-tonteras/blob/rally-uno/Rally%201.ipynb)

Corte de datos: **24 de septiembre de 2026**.

In [ ]:
!pip -q install yfinance statsmodels


In [ ]:
import os, random, warnings
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("Seed:", SEED)


## 1. Datos

In [ ]:
START = "2020-01-01"
END = "2026-09-25"

tickers = ["MXN=X", "^GSPC", "^VIX", "^TNX"]
names = {"MXN=X":"USDMXN", "^GSPC":"SP500", "^VIX":"VIX", "^TNX":"TNX10Y"}

raw = yf.download(
    tickers, start=START, end=END,
    auto_adjust=False, progress=False, threads=False
)

if raw.empty:
    raise RuntimeError("Yahoo Finance no devolvió datos.")

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw.copy()

prices = prices.rename(columns=names)
cols = ["USDMXN", "SP500", "VIX", "TNX10Y"]
missing = [c for c in cols if c not in prices.columns]
if missing:
    raise RuntimeError(f"Faltan columnas: {missing}")

prices = prices[cols].apply(pd.to_numeric, errors="coerce").dropna().sort_index()
prices = prices.loc[~prices.index.duplicated(keep="first")]

assert len(prices) >= 500
assert prices.index.is_unique
assert prices.index.is_monotonic_increasing
assert prices.index.max() <= pd.Timestamp("2026-09-24")

data = pd.DataFrame(index=prices.index)
data["usdmxn_ret"] = 100*np.log(prices["USDMXN"]).diff()
data["sp500_ret"] = 100*np.log(prices["SP500"]).diff()
data["vix_change"] = prices["VIX"].diff()
data["tnx_change"] = prices["TNX10Y"].diff()
data = data.replace([np.inf, -np.inf], np.nan).dropna()

display(data.describe().T)
display(data.corr())

def adf_row(s, regression):
    r = adfuller(s.dropna(), regression=regression, autolag="AIC")
    return {"ADF":r[0], "p_value":r[1], "rezagos":r[2], "n_obs":r[3]}

adf = pd.DataFrame({
    "USD/MXN nivel": adf_row(prices["USDMXN"], "ct"),
    "USD/MXN rendimiento": adf_row(data["usdmxn_ret"], "c"),
    "S&P 500 nivel": adf_row(prices["SP500"], "ct"),
    "S&P 500 rendimiento": adf_row(data["sp500_ret"], "c")
}).T
display(adf)


## 2. Modelos

In [ ]:
cut = int(len(data)*0.80)
train = data.iloc[:cut].copy()
test = data.iloc[cut:].copy()

assert train.index.max() < test.index.min()

test_dates = test.index.copy()
ytr = train["usdmxn_ret"].reset_index(drop=True).astype(float)
yte = test["usdmxn_ret"].reset_index(drop=True).astype(float)

x = ["sp500_ret", "vix_change", "tnx_change"]
Xtr = train[x].reset_index(drop=True).astype(float)
Xte = test[x].reset_index(drop=True).astype(float)

assert ytr.index.equals(Xtr.index)
assert yte.index.equals(Xte.index)

def metricas(y, pred):
    y = np.asarray(y, float)
    pred = np.asarray(pred, float)
    if len(y) != len(pred):
        raise ValueError("Longitudes distintas.")
    return {
        "MAE": float(mean_absolute_error(y, pred)),
        "RMSE": float(mean_squared_error(y, pred)**0.5)
    }

# ARIMA: selección por AIC sólo con entrenamiento
arima_rows = []
for p in range(4):
    for q in range(4):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                fit = ARIMA(ytr, order=(p,0,q), trend="c").fit()
            if bool(fit.mle_retvals.get("converged", True)) and np.isfinite(fit.aic):
                arima_rows.append((p,q,float(fit.aic)))
        except Exception:
            pass

arima_tab = pd.DataFrame(arima_rows, columns=["p","q","AIC"]).sort_values("AIC").reset_index(drop=True)
if arima_tab.empty:
    raise RuntimeError("No convergió ningún ARIMA.")

bp, bq = int(arima_tab.loc[0,"p"]), int(arima_tab.loc[0,"q"])
arima_order = (bp,0,bq)
ar = ARIMA(ytr, order=arima_order, trend="c").fit()
ar_pred = np.asarray(ar.forecast(steps=len(yte)), float)

# SARIMAX: selección independiente por AIC sólo con entrenamiento
sx_rows, sx_fits = [], {}
for p in range(4):
    for q in range(4):
        order = (p,0,q)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                fit = SARIMAX(
                    ytr, exog=Xtr, order=order, trend="c",
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(disp=False, maxiter=1000)
            ok = bool(fit.mle_retvals.get("converged", True))
            ok = ok and np.isfinite(fit.aic) and np.isfinite(np.asarray(fit.params, float)).all()
            if ok:
                sx_rows.append((p,q,float(fit.aic)))
                sx_fits[order] = fit
        except Exception:
            pass

sx_tab = pd.DataFrame(sx_rows, columns=["p","q","AIC"]).sort_values("AIC").reset_index(drop=True)
if sx_tab.empty:
    raise RuntimeError("No convergió ningún SARIMAX.")

sp, sq = int(sx_tab.loc[0,"p"]), int(sx_tab.loc[0,"q"])
sx_order = (sp,0,sq)
sx = sx_fits[sx_order]
sx_pred = np.asarray(sx.get_forecast(steps=len(yte), exog=Xte).predicted_mean, float)

ar_pred = pd.Series(ar_pred, index=test_dates)
sx_pred = pd.Series(sx_pred, index=test_dates)
zero = pd.Series(np.zeros(len(yte)), index=test_dates)

comparacion = pd.DataFrame([
    {"modelo":"Benchmark cero", **metricas(yte, zero)},
    {"modelo":f"ARIMA{arima_order}", **metricas(yte, ar_pred)},
    {"modelo":f"SARIMAX{sx_order} condicional", **metricas(yte, sx_pred)}
]).set_index("modelo")

print("ARIMA:", arima_order)
print("SARIMAX:", sx_order)
display(arima_tab.head(10))
display(sx_tab.head(10))
display(comparacion)
display(acorr_ljungbox(ar.resid, lags=[10,20], return_df=True))

n = min(150, len(test))
plt.figure(figsize=(13,5))
plt.plot(test_dates[-n:], test["usdmxn_ret"].iloc[-n:], label="Observado")
plt.plot(ar_pred.index[-n:], ar_pred.iloc[-n:], label="ARIMA")
plt.plot(sx_pred.index[-n:], sx_pred.iloc[-n:], label="SARIMAX condicional")
plt.axhline(0, linewidth=.8)
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 3. Sensibilidad

In [ ]:
reg = data.copy()
reg["lag_usdmxn"] = reg["usdmxn_ret"].shift(1)
reg = reg.dropna()

X = sm.add_constant(reg[["sp500_ret","vix_change","tnx_change","lag_usdmxn"]])
ols = sm.OLS(reg["usdmxn_ret"], X).fit(
    cov_type="HAC",
    cov_kwds={"maxlags":5}
)

print(ols.summary())

beta = float(ols.params["sp500_ret"])
pval = float(ols.pvalues["sp500_ret"])
ci = ols.conf_int().loc["sp500_ret"]

display(pd.DataFrame({
    "coeficiente":[beta],
    "p_value":[pval],
    "IC95_inf":[float(ci.iloc[0])],
    "IC95_sup":[float(ci.iloc[1])]
}, index=["S&P 500"]))

rolling_beta = (
    data["usdmxn_ret"].rolling(60).cov(data["sp500_ret"])
    / data["sp500_ret"].rolling(60).var()
).replace([np.inf,-np.inf], np.nan)

plt.figure(figsize=(13,4))
plt.plot(rolling_beta.index, rolling_beta)
plt.axhline(0, linewidth=.8)
plt.grid(alpha=.25)
plt.show()


## 4. Pronóstico y validación

In [ ]:
yfull = data["usdmxn_ret"].reset_index(drop=True).astype(float)
final_ar = ARIMA(yfull, order=arima_order, trend="c").fit()

fc = final_ar.get_forecast(steps=5).summary_frame(alpha=.05)[
    ["mean","mean_ci_lower","mean_ci_upper"]
].copy()
fc.columns = ["pronostico_pct","limite_inferior_95","limite_superior_95"]
display(fc)

checks = pd.Series({
    "seed": SEED == 42,
    "sin_NA": not data.isna().any().any(),
    "corte": prices.index.max() <= pd.Timestamp("2026-09-24"),
    "train_test": train.index.max() < test.index.min(),
    "indices_train": ytr.index.equals(Xtr.index),
    "indices_test": yte.index.equals(Xte.index),
    "arima_longitud": len(ar_pred) == len(yte),
    "sarimax_longitud": len(sx_pred) == len(yte),
    "metricas_finitas": np.isfinite(comparacion[["MAE","RMSE"]].to_numpy()).all(),
    "beta_finita": np.isfinite(beta),
    "forecast_finito": np.isfinite(fc.to_numpy()).all()
}, name="OK")

display(checks)
if not bool(checks.all()):
    raise AssertionError("Falló una validación.")

print("Validaciones superadas.")
